# Multi-Disease Medical AI — Step 1 & 2: Dataset Verification & DIP Analysis
### ECE Major Project
**Project Title:** AI-Based Multi-Disease Detection from Medical Images Using Digital Image Processing and Deep Learning

This notebook verifies MedMNIST installation, downloads the $224 \times 224$ versions of **ChestMNIST**, **DermaMNIST**, and **RetinaMNIST**, and demonstrates the **Digital Image Processing (DIP)** pipeline.

In [ ]:
# STEP 1: Install MedMNIST and Core Imaging Dependencies
!pip install -q medmnist torchvision scikit-learn opencv-python-headless matplotlib

In [ ]:
# STEP 2: Verify Setup
import medmnist
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt

print("MedMNIST version:", medmnist.__version__)
print("PyTorch version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
print("Environment verification successful!")

## 1. Download & Verify the Three Medical Imaging Modalities (224x224)
- **🫁 ChestMNIST**: Multi-label Chest X-Ray (14 pathologies)
- **🩺 DermaMNIST**: Multi-class Dermoscopic Skin Lesions (7 classes)
- **👁️ RetinaMNIST**: Ordinal Retinal Fundus (5 Diabetic Retinopathy grades)

In [ ]:
from medmnist import ChestMNIST, DermaMNIST, RetinaMNIST

DATA_DIR = "./data"
IMAGE_SIZE = 224  # High-resolution MedMNIST+

print("Downloading / Loading ChestMNIST...")
chest_train = ChestMNIST(split="train", download=True, size=IMAGE_SIZE, root=DATA_DIR)
chest_test  = ChestMNIST(split="test", download=True, size=IMAGE_SIZE, root=DATA_DIR)

print("Downloading / Loading DermaMNIST...")
derma_train = DermaMNIST(split="train", download=True, size=IMAGE_SIZE, root=DATA_DIR)
derma_test  = DermaMNIST(split="test", download=True, size=IMAGE_SIZE, root=DATA_DIR)

print("Downloading / Loading RetinaMNIST...")
retina_train = RetinaMNIST(split="train", download=True, size=IMAGE_SIZE, root=DATA_DIR)
retina_test  = RetinaMNIST(split="test", download=True, size=IMAGE_SIZE, root=DATA_DIR)

print(f"\nDataset Sizes:")
print(f"  ChestMNIST  : Train={len(chest_train)}, Test={len(chest_test)}")
print(f"  DermaMNIST  : Train={len(derma_train)}, Test={len(derma_test)}")
print(f"  RetinaMNIST : Train={len(retina_train)}, Test={len(retina_test)}")

## 2. Visualize Medical Images Across Modalities
Displaying sample images and labels from each modality in a clear grid.

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 12))
plt.subplots_adjust(wspace=0.3, hspace=0.3)

# Row 0: Chest X-Rays
for i in range(4):
    img, target = chest_train[i]
    axes[0, i].imshow(img, cmap="gray")
    axes[0, i].set_title(f"Chest X-Ray #{i}")
    axes[0, i].axis("off")

# Row 1: Skin Dermoscopy
for i in range(4):
    img, target = derma_train[i]
    axes[1, i].imshow(img)
    axes[1, i].set_title(f"Skin Lesion #{i} (Class {target[0]})")
    axes[1, i].axis("off")

# Row 2: Retinal Fundus
for i in range(4):
    img, target = retina_train[i]
    axes[2, i].imshow(img)
    axes[2, i].set_title(f"Retina #{i} (DR Grade {target[0]})")
    axes[2, i].axis("off")

plt.suptitle("Sample Medical Images from ChestMNIST, DermaMNIST & RetinaMNIST (224x224)", fontsize=16, y=0.95)
plt.show()

## 3. ECE Digital Image Processing (DIP) Pipeline
Demonstrating:
1. **Chest DIP**: Bilateral Denoising + CLAHE (expands dynamic range across lung & mediastinum)
2. **Skin DIP**: DullRazor Hair Removal (morphological top-hat + inpainting) + Shades-of-Gray Color Constancy
3. **Retina DIP**: Ben Graham Illumination Normalization (local color subtraction) + Green Channel CLAHE

In [ ]:
# Example Chest DIP Implementation
chest_sample = np.array(chest_train[0][0])
if len(chest_sample.shape) == 3:
    chest_gray = cv2.cvtColor(chest_sample, cv2.COLOR_RGB2GRAY)
else:
    chest_gray = chest_sample

# Step 1: Bilateral Filter (edge-preserving denoising)
denoised = cv2.bilateralFilter(chest_gray, d=9, sigmaColor=75, sigmaSpace=75)
# Step 2: CLAHE (Adaptive Histogram Equalization)
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
enhanced = clahe.apply(denoised)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(chest_gray, cmap="gray")
axes[0].set_title("Raw Chest X-Ray")
axes[0].axis("off")

axes[1].imshow(denoised, cmap="gray")
axes[1].set_title("Bilateral Filtered (Denoised)")
axes[1].axis("off")

axes[2].imshow(enhanced, cmap="gray")
axes[2].set_title("CLAHE Enhanced (Dynamic Range Expanded)")
axes[2].axis("off")

plt.tight_layout()
plt.show()